# B → Xᵤℓν hybrid model — example

This notebook walks through computing both the optimal transport (OT) hybrid weights
and the conventional bin-by-bin hybrid weights for a single decay mode.

Set the paths in the **Configuration** cell below, then run all cells in order.

In [ ]:
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import yaml

from hybrid import (
    build_phase_space_grid,
    solve_ot,
    extract_reweights,
    apply_ot_weights,
    plot_sink_distribution,
    compute_normalization,
    compute_conventional_weights,
    apply_bin_weights,
    compute_all_moments,
    plot_moment_comparison,
    print_moment_errors,
)

## Configuration

You can either point to your `config.yaml` or set the parameters directly here.

**Required columns in your ROOT/parquet files:**

| Column | Description |
|--------|-------------|
| `genPplus` | P+ = E_X + |p_X| in the B rest frame [GeV] |
| `genPminus` | P- = E_X − |p_X| in the B rest frame [GeV] |
| `X_gen_PDG` | PDG code of the hadronic system |
| `FF_weight` | Form-factor weight (1 if not applicable) |
| `genMx` | Hadronic invariant mass [GeV] |
| `genq2` | q² [GeV²] |
| `gen_lep_E_B` | Lepton energy in B rest frame [GeV] |
| `genCosThetaL` | cos θ_ℓ |

In [ ]:
# ← set your paths here
PATH_INCLUSIVE = "/path/to/inclusive.root"
PATH_RESONANT  = "/path/to/resonant.root"

MODE = "charged"   # "charged" (B+) or "neutral" (B0)
INCL_MODEL = "DFN" # label for plots

# Branching fractions (absolute values)
BFS = {
    111: 7.8e-4,    # π⁰
    221: 3.5e-4,    # η
    113: 15.8e-4,   # ρ⁰
    223: 11.9e-4,   # ω
    331: 2.4e-4,    # η'
} if MODE == "charged" else {
    211: 15.0e-4,   # π±
    213: 29.4e-4,   # ρ±
}
INCL_BF = 19.2e-4  # B(B → Xᵤℓν)

BIN_WIDTH = 0.08   # GeV

## 1. Load data

In [ ]:
columns = [
    "X_gen_PDG", "genPplus", "genPminus",
    "genMx", "genq2", "gen_lep_E_B", "genCosThetaL",
    "FF_weight",
]

def read_root(path, cols):
    with uproot.open(path) as f:
        tree_name = next(k.split(";")[0] for k in f.keys())
        return f[tree_name].arrays(cols, library="pd")

df_incl = read_root(PATH_INCLUSIVE, columns)
df_excl = read_root(PATH_RESONANT,  columns)

# FF_weight = 1 if the column is absent or you don't use it
df_incl["FF_weight"] = df_incl.get("FF_weight", pd.Series(1.0, index=df_incl.index))
df_excl["FF_weight"] = df_excl.get("FF_weight", pd.Series(1.0, index=df_excl.index))

print(f"Inclusive events: {len(df_incl):,}")
print(f"Resonant events:  {len(df_excl):,}")

## 2. Normalization weights

The exclusive sample needs to be scaled so that the combined hybrid model integrates to the
correct total inclusive branching fraction. The normalization factor is:

$$w_\mathrm{excl} = \frac{N_\mathrm{incl}}{N_\mathrm{excl}} \left(1 - \frac{\mathcal{B}_{X_u}}{\sum_i \mathcal{B}_i + \mathcal{B}_{X_u}}\right)$$

This adds a `total_weight` column to both DataFrames.

In [ ]:
compute_normalization(
    df_incl, df_excl,
    branching_fractions=BFS,
    inclusive_bf=INCL_BF,
)
print("excl normalization weight:", df_excl["branching_fraction_weight"].iloc[0])

## 3. OT hybrid weights

We build 2D histograms in (P+, P-) space for both samples, solve the optimal transport
problem to find the minimal-cost redistribution, and extract per-bin reweights.

In [ ]:
Pinc, Pres = build_phase_space_grid(
    df_incl, df_excl,
    bin_width=BIN_WIDTH,
    pminus_range=(0, 5.31),
    pplus_range=(0, 3.01),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, dist, title in zip(axes, [Pinc, Pres], ["Inclusive", "Resonant"]):
    im = ax.imshow(dist, origin="lower", aspect="auto", cmap="magma",
                   extent=[0, 5.31, 0, 3.01])
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(r"$P^-$ [GeV]")
    ax.set_ylabel(r"$P^+$ [GeV]")
    plt.colorbar(im, ax=ax)
plt.tight_layout()

In [ ]:
G, src_coords, src_shape = solve_ot(
    Pinc, Pres,
    bin_width=BIN_WIDTH,
    window=None,      # no spatial restriction
    lambda_reg=0,     # exact EMD
    verbose=True,
)

reweight_map = extract_reweights(G, src_coords, src_shape, Pinc)
df_incl["ot_hybrid_weight"] = apply_ot_weights(df_incl, reweight_map, BIN_WIDTH)

In [ ]:
# Inspect the weight map
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(reweight_map, origin="lower", aspect="auto", cmap="viridis",
               extent=[0, 5.31, 0, 3.01], vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Weight")
ax.set_xlabel(r"$P^-$ [GeV]")
ax.set_ylabel(r"$P^+$ [GeV]")
ax.set_title("OT hybrid reweights")
plt.tight_layout()

## 4. Conventional bin-by-bin hybrid (for comparison)

In [ ]:
binning = {
    "genq2":       np.array([0, 2.5, 5, 7.5, 10, 12.5, 15, 20, 25]),
    "gen_lep_E_B": np.array([0, 0.5, 1, 1.25, 1.5, 1.75, 2, 2.25, 3]),
    "genMx":       np.array([0, 1.4, 1.6, 1.8, 2, 2.5, 3, 3.5]),
}

conv_arr = compute_conventional_weights(df_incl, df_excl, binning, weight_col="total_weight")
df_incl["conventional_hybrid_weight"] = apply_bin_weights(df_incl, binning, conv_arr)

## 5. Kinematic distributions

Compare inclusive DFN vs. OT hybrid vs. conventional hybrid for the four main variables.

In [ ]:
from plothist import get_color_palette

variables = [
    ("genMx",        np.linspace(0, 3.51, 50),  r"$M_X$ [GeV]"),
    ("gen_lep_E_B",  np.linspace(0, 2.65, 50),  r"$E_\ell^B$ [GeV]"),
    ("genq2",        np.linspace(0, 25.01, 50), r"$q^2$ [GeV$^2$]"),
    ("genCosThetaL", np.linspace(-1, 1, 50),    r"$\cos\theta_\ell$"),
]

if MODE == "charged":
    masks = {
        r"$\pi^0$":  df_excl["X_gen_PDG"].abs() == 111,
        r"$\eta$":   df_excl["X_gen_PDG"].abs() == 221,
        r"$\rho^0$": df_excl["X_gen_PDG"].abs() == 113,
        r"$\omega$": df_excl["X_gen_PDG"].abs() == 223,
        r"$\eta'$":  df_excl["X_gen_PDG"].abs() == 331,
    }
else:
    masks = {
        r"$\pi^\pm$": df_excl["X_gen_PDG"].abs() == 211,
        r"$\rho^\pm$":df_excl["X_gen_PDG"].abs() == 213,
    }

palette = get_color_palette("cubehelix", len(masks) + 1)[::-1]
def rgba(k):
    r, g, b = palette[k][:3]
    return (r, g, b, 0.85)
all_labels = ["Incl leftover"] + list(masks.keys())
all_colors = [rgba(0)] + [rgba(j + 1) for j in range(len(masks))]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
mode_label = r"$B^\pm$" if MODE == "charged" else r"$B^0$"

for i, (var, bins, xlabel) in enumerate(variables):
    ax = axes[i]

    h_dfn  = np.histogram(df_incl[var], bins=bins, weights=df_incl["total_weight"])[0]
    h_incl = np.histogram(df_incl[var], bins=bins,
                           weights=df_incl["total_weight"] * df_incl["ot_hybrid_weight"])[0]
    h_excl = [np.histogram(df_excl[var][m], bins=bins,
                            weights=df_excl["total_weight"][m])[0]
              for m in masks.values()]

    bottom = np.zeros_like(h_incl, dtype=float)
    for hdat, lab, col in zip([h_incl] + h_excl, all_labels, all_colors):
        ax.bar(bins[:-1], hdat, width=np.diff(bins), bottom=bottom,
               align="edge", color=col, edgecolor="none", label=lab)
        bottom += hdat

    ax.step(bins[:-1], bottom, where="post", color="k", lw=1.5, label="OT hybrid")
    ax.step(bins[:-1], h_dfn,  where="post", color="r", lw=1.2, label=INCL_MODEL)
    ax.text(0.05, 0.95, mode_label, transform=ax.transAxes, va="top", ha="left", fontsize=14)
    if i == 0:
        ax.legend(fontsize=11)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel("Events", fontsize=13)
    ax.grid(alpha=0.3)

plt.tight_layout()

## 6. Moment comparison

How well does each method preserve the HQE spectral moments?

In [ ]:
for df in (df_incl, df_excl):
    df["genMxSq"] = df["genMx"] ** 2
df_excl["_null"] = 0.0
df_incl["_conv_wt"] = df_incl["total_weight"] * df_incl["conventional_hybrid_weight"]

varlist = ["genMxSq", "genq2", "gen_lep_E_B"]
kw = dict(df_incl=df_incl, df_excl=df_excl, varlist=varlist, n=4)

mom_ref      = compute_all_moments(**kw, incl_weight_col="total_weight",  excl_weight_col="_null",      central=False)
mom_ref_cen  = compute_all_moments(**kw, incl_weight_col="total_weight",  excl_weight_col="_null",      central=True)
mom_ot       = compute_all_moments(**kw, incl_weight_col="ot_hybrid_weight",   excl_weight_col="total_weight", central=False)
mom_ot_cen   = compute_all_moments(**kw, incl_weight_col="ot_hybrid_weight",   excl_weight_col="total_weight", central=True)
mom_conv     = compute_all_moments(**kw, incl_weight_col="_conv_wt",      excl_weight_col="total_weight", central=False)
mom_conv_cen = compute_all_moments(**kw, incl_weight_col="_conv_wt",      excl_weight_col="total_weight", central=True)

print_moment_errors(mom_ot, mom_conv, mom_ref, mom_ot_cen, mom_conv_cen, mom_ref_cen, varlist)

In [ ]:
fig = plot_moment_comparison(
    mom_ot, mom_conv, mom_ref,
    mom_ot_cen, mom_conv_cen, mom_ref_cen,
    varlist, MODE,
)